# Data Science for Business - Logit on Micro Mortgages

*Session 4 · ISLP 4.3 · 4.4.2 (Tables 4.4–4.7, Fig. 4.8 ROC) · 5.1.3, 5.1.5*

## Initialize notebook
Load required packages. Set up workspace, e.g., set theme for plotting and initialize the random number generator.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, RocCurveDisplay, roc_auc_score
from sklearn.linear_model import LogisticRegression
import statsmodels.formula.api as smf

In [ ]:
np.random.seed(42)
plt.style.use('fivethirtyeight')

## Problem description

In India, there are about 20 million home loan (mortgage) aspirants
working in the informal sector:

- Monthly income between INR 20,000-25,000 (\$ 325-400)
- Typically no formal accounts and documents (e.g., tax returns, income proofs, bank statements)
- Often use services of money lenders with interest rates between 30 and 60% per annum

Providing mortgages to this group of customers requires to quickly and
efficiently assess their creditworthiness. Due to a lack of formal
documents and objective data, most financial institutions perform
interview-based processes to decide about these loan requests:

Strength of the current process:

-   Interview-based field assessment

-   Relaxation of document requirements

Weaknesses of the current process:

-   Costly (total transaction costs as high as 30% of loan volume)

-   Subjective judgments; depends on individual skills and motivations

-   Low reliability across branches and credit officers

-   Risk of corruption and fraud

The data comes from a Harvard Business Review case. Our goal: predict the credit officers' decision (`Decision`: 1 = approved, 0 = rejected) from the application data. See `variables.pdf` for a description of the variables.

## Load data

Load data from CSV file.

In [ ]:
data = pd.read_csv('https://raw.githubusercontent.com/olivermueller/ds4b-2026/main/Session_04/micromortgage.csv')

In [ ]:
data.head()

## Prepare data

We do some minimal data preparation, that is, drop the `ID` column and turn the `Tier` variable into a string, so that it is treated as categorical.

In [ ]:
data = data.drop(['ID'], axis=1)
data["Tier"] = "T" + data["Tier"].astype(str)

Next, we perform a standard 80:20 random train/test split (validation set approach, ISLP 5.1.1). `stratify` keeps the share of approvals equal in both sets.

In [ ]:
train, test = train_test_split(data, test_size=0.2, random_state=42, stratify=data["Decision"])

## Exploratory data analysis

### Descriptive summary statistics

Calculate the base rate of mortgage approvals.

In [ ]:
train["Decision"].mean()

About 80% of applications are approved. That's the benchmark a useful model must beat: always predicting "approve" already gives 80% accuracy.

Note that here class 1 is *approval*, whereas the positive class in ISLP Ch. 4 is *default*. Keep this in mind when we discuss precision and recall below.

### Relationships between response and predictors

In [ ]:
sns.boxplot(data=train, x="Decision", y="Age")
plt.show()

**Your task:** Plot `TotInc` (total income) against `Decision`.

In [ ]:
# YOUR CODE HERE

In [ ]:
sns.barplot(data=train, x="Gender", y="Decision")
plt.show()

**Your task:** Compare approval rates by `Tier` or `Employment_Type`.

In [ ]:
# YOUR CODE HERE

## Fit model

Ok, now we are ready to fit our first logistic regression model (ISLP 4.3). We use `Age`, `Gender`, `Dwnpay` (down payment), `LoanReq` (requested loan amount) and `Term` as predictors, plus the interaction between `LoanReq` and `Term`. `Gender` is coded with `Female` as the baseline (alphabetically first).

In [ ]:
model_logit = smf.logit(formula='Decision ~ Age + Gender + Dwnpay + LoanReq * Term', data=train)
model_logit = model_logit.fit()

In [ ]:
print(model_logit.summary())

## Make predictions

Let's compute predictions, both in the form of probabilities and binary decisions. We classify an application as approved if the predicted probability is at least 0.5. A bank would pick the threshold from the cost of a wrong approval vs. a wrong rejection (ISLP 4.4.2).

In [ ]:
threshold = 0.5
pred_proba = model_logit.predict(test)
pred_label = (pred_proba >= threshold).astype(int)

In [ ]:
pred_proba

In [ ]:
pred_label

## Evaluate accuracy

We begin by calculating some standard metrics like precision, recall, F1-score, and accuracy. Remember that the row for class 1 refers to *approvals*.

`zero_division=0` reports a precision of 0 instead of a warning if the model never predicts a class.

In [ ]:
print(classification_report(test["Decision"], pred_label, zero_division=0))

With a threshold of 0.5, our model approves (almost) every application: its accuracy is no better than the base rate, and the recall for rejections (class 0) is close to zero.

The confusion matrix helps us to diagnose what kind of mistakes the model makes. Careful: scikit-learn puts the **true** classes in the rows and the predicted classes in the columns, i.e., it is transposed compared to ISLP Tables 4.4 and 4.6.

In [ ]:
ConfusionMatrixDisplay.from_predictions(test["Decision"], pred_label)
plt.grid(False)
plt.show()

Finally, we plot the ROC curve and compute the AUC, which characterizes the performance of a classifier over all thresholds in one single number.

In [ ]:
RocCurveDisplay.from_predictions(test["Decision"], pred_proba, name="Logistic regression")
plt.show()

print("Test AUC:", roc_auc_score(test["Decision"], pred_proba))

## Cross-validation

### Why cross-validation?

Our test AUC above depends on which rows happened to land in the test set; with another random split, we would get a somewhat different number. $k$-fold cross-validation (ISLP 5.1.3) splits the data into $k$ folds and fits the model $k$ times, each time holding out a different fold for evaluation and training on the remaining $k-1$ folds. Averaging the $k$ results gives a more robust estimate, and their spread shows how much it varies. With $k=5$:

```
Fold 1: [Test ] [Train] [Train] [Train] [Train]
Fold 2: [Train] [Test ] [Train] [Train] [Train]
Fold 3: [Train] [Train] [Test ] [Train] [Train]
Fold 4: [Train] [Train] [Train] [Test ] [Train]
Fold 5: [Train] [Train] [Train] [Train] [Test ]
```

### Select predictors and response

We run the cross-validation on the training data only, so the test set stays untouched. Note that this is a different model than above: we use a set of numerical loan characteristics as predictors, and scikit-learn instead of statsmodels.

In [ ]:
X_cv = train[['LoanReq', 'Term', 'Dwnpay', 'BankSave', 'CalcEmi', 'IIR', 'IAR', 'FOIR', 'LTV', 'LVR']]
y_cv = train['Decision']

### Define the model

This time we use scikit-learn's `LogisticRegression` instead of statsmodels. It is the same kind of model as above. The two settings in the code below only make sure that scikit-learn fits the plain logistic regression from the lecture: `C=1e10` switches off an extra penalty that scikit-learn adds by default, and `solver='newton-cholesky'` chooses a fitting method that works reliably with our data. You don't need to worry about the technical details.

In [ ]:
model_cv = LogisticRegression(C=1e10, solver='newton-cholesky')
model_cv

### Define the folds

`StratifiedKFold` creates folds that all have (about) the same approval rate as the training data (*stratified*). `shuffle=True` assigns the rows to folds at random instead of in their original order; `random_state=42` makes this reproducible.

In [ ]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Run the cross-validation

`cross_val_score` fits the model five times (each time on four folds) and returns one score per held-out fold; with `scoring='roc_auc'` that is the AUC.

In [ ]:
cv_scores = cross_val_score(model_cv, X_cv, y_cv, cv=folds, scoring='roc_auc')

print('AUC per fold:', cv_scores.round(4))
print(f'Mean AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f} (standard deviation)')

### Interpret the results

The mean AUC across the five folds is our estimate of how well this model performs on new data; the standard deviation shows how much the AUC varies from fold to fold, i.e., how much a single split could mislead us. Compare the mean with the test AUC of our first model above. Keep in mind that the two models use different predictors: the loan characteristics (e.g., the ratios `FOIR` and `LTV`) seem to carry much more information about the decision than age and gender.

## Your turn!

**Your task:** Improve the logistic regression model from *Fit model* by including more predictors (e.g., `Tier`, `Employment_Type`, `TotInc`, `LTV`). Did the extra predictors improve the test AUC, or did you overfit?

In [ ]:
# YOUR CODE HERE